[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/induced_charge/induced_charge.ipynb)

# Induced Surface Charge

This example simulates how conserved charge migrates from the interiors of isolated conductors towards their surfaces under a turning external electric field. A negatively charged ring and a positively charged rectangular rod sit inside an insulator with a smooth conductivity transition across each boundary, so conduction redistributes charge inside each body without transferring it across the insulating gaps.

On a triangulated planar domain in projective spacetime, each vertex's ***simplex gradient*** is wedged with the time covector `-(gradients ^ t)` to form the corner electric field as an extrinsic bivector. Pairing that field with the spatial worldsheet response `field & electric_response(field)` supplies both electric displacement when scaled by permittivity and Ohmic conduction when scaled by conductivity, while the commutator `field.commutator(time)` gives the force on a unit rest charge. See the [COMSOL electroquasistatics overview](https://www.comsol.com/multiphysics/electroquasistatics-theory).

In Gibbs–Heaviside vector notation and finite-element notation, electroquasistatic relaxation solves the coupled divergence equations $-\nabla \cdot (\varepsilon \nabla \phi) = \rho$ and $\partial_t \rho - \nabla \cdot (\sigma \nabla \phi) = 0$ with electric field $\mathbf{E} = -\nabla \phi$, assembling the stiffness and conduction matrices $K_{ij} = \sum_e \int_{\Omega_e} \varepsilon \, (J_e^{-T} \nabla_{\hat{x}} \hat{N}_i) \cdot (J_e^{-T} \nabla_{\hat{x}} \hat{N}_j) \, |\det J_e| \, d\hat{x}$ from element reference Jacobians $J_e$ and hat functions $\hat{N}_i$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"


In [ ]:
from collections.abc import Iterator
import numpy as np
from numga import Algebra, NumpyContext, stack, where
from numga.sparse import Sparse, spdiag
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.electromagnetism.induced_charge import render, scenarios

ga = Algebra("x+y+t-w0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                     # Scalar+
Upright = ga.gatype.vector().derive.within(Algebra("x+y+"))     # Vector+
Point = ga.gatype.antivector()                                  # Antivector-
Field = ga.gatype.bivector()                                    # Bivector+
Excitation = ga.gatype.antibivector()                           # Antibivector-
origin = mv.w.dual()                                            # Point-
time = -mv.t.dual()                                             # Point-
axes = stack((mv.x.dual(), mv.y.dual()))                        # [2] Point-
sweeps = axes & time                                            # [2] Excitation-
electric_response = (sweeps * (Field & sweeps)).sum(axis=0)     # [] Excitation- <- Field+


## Isolated conductors in an insulator

A negatively charged ring and a positively charged rectangular rod sit inside an insulating square whose outer boundary supplies the turning applied potential. Each conductor's material fraction falls smoothly to zero across a thin boundary layer, keeping the surrounding insulator at zero conductivity.

In [ ]:
n_across, n_frames = scenarios.n_across, scenarios.n_frames
permittivity, conductivities = scenarios.permittivity, scenarios.conductivities
body_charges = scenarios.body_charges
frame_step, n_substeps = scenarios.frame_step, scenarios.n_substeps
frame_duration_ms = scenarios.frame_duration_ms
topology, vertices, fractions, memberships, boundary = scenarios.conductor_scene(n_across, n_frames)
metal = fractions.sum(axis=0)                                   # Scalar+[V]
save_figure(render.setup_figure(topology, vertices, metal), "charge_relaxation_notebook_setup");


Lighter grey regions mark the ring and the tilted rectangular rod, bounded by contour curves halfway across each conductivity transition.

## Electric storage and Ohmic conduction

Spatial displacements swept along the time direction form intrinsic worldsheet patches `sweeps = axes & time`. Pairing an electric field bivector against `electric_response` reads those patches: scaled by permittivity across each triangle it gives the vertex-to-vertex capacitance map, and scaled by local conductivity it gives the Ohmic outflow map.

In [ ]:
def triangle_geometry(topology: Topology, vertices: Point) -> tuple[Scalar, Scalar, Field]:
    """Triangle areas (`Scalar+[F]`), vertex dual areas (`Scalar+[V]`), and simplex-gradient electric fields (`[3] Field+[F]`) come from the vertex positions."""
    corners = topology.incidence[2, 0]                          # [3] Scalar+[F <- V]
    corner_mean = corners.mean(axis=0)                          # Scalar+[F <- V]
    corner_vertices = corners * vertices                        # [3] Point-[F]
    double_areas = (corner_vertices[0] & corner_vertices[1] & corner_vertices[2] & time).norm()  # Scalar+[F]
    areas = double_areas / 2                                    # Scalar+[F]
    dual_areas = corner_mean.adjugate() * areas                 # Scalar+[V]
    facing = topology.off[2, 0] * vertices                      # [2, 3] Point-[F]
    sides = facing[0] & facing[1]                               # [3] Excitation-[F]
    gradients = ((sides & time) / (sides & time & corner_vertices)).cast(Upright)  # [3] Upright+[F]
    corner_fields = -(gradients ^ mv.t)                         # [3] Field+[F]
    return areas, dual_areas, corner_fields


def surface_operators(topology: Topology, areas: Scalar, corner_fields: Field, fractions: Scalar,
                      permittivity: float, conductivities: np.ndarray) -> tuple[Sparse, Sparse, Sparse, Scalar]:
    """Permittivity and conductivity weight the same electric-field pairing to form the vertex charge and outflow maps, with interior Poisson operator `(Scalar+ <- Scalar+)[V_interior <- V_interior]`."""
    corners = topology.incidence[2, 0]                          # [3] Scalar+[F <- V]
    corner_mean = corners.mean(axis=0)                          # Scalar+[F <- V]
    # Displacement stores electric energy while conduction dissipates it through the same field pairing.
    local_energy = areas * (corner_fields[:, None] & electric_response(corner_fields[None, :]))  # [3, 3] Scalar+[F]
    conductivity = (conductivities * (corner_mean * fractions)).sum(axis=0)  # Scalar+[F]
    charge_map = (corners[:, None].adjugate() * spdiag(permittivity * local_energy)
                  * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]
    outflow_map = (corners[:, None].adjugate() * spdiag(conductivity * local_energy)
                   * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
    extension = topology.reading(topology.interior[0], 0).adjugate()  # Scalar+[V <- V_interior]
    poisson = (extension.adjugate() * charge_map * extension) * Scalar  # (Scalar+ <- Scalar+)[V_interior <- V_interior]
    return charge_map, outflow_map, extension, poisson


## Charge relaxation in a turning field

Each conductor begins with its excess charge distributed over its dual areas. Conduction drives charge outward until the self-field cancels the interior field; turning the outer boundary field through half a revolution disturbs that balance and drives a fresh redistribution. Computing and rendering the 100 frames takes several seconds.

In [ ]:
def relaxation_states(topology: Topology, vertices: Point, fractions: Scalar, boundary: Scalar,
                      permittivity: float, conductivities: np.ndarray, body_charges: np.ndarray,
                      frame_step: float, n_substeps: int) -> Iterator[tuple[Scalar, Scalar, Scalar, Point]]:
    """Yields the potential (`Scalar+[V]`), integrated vertex charge (`Scalar+[V]`), charge density (`Scalar+[V]`), and rest-charge force (`Point-[F]`) across the relaxation."""
    corners = topology.incidence[2, 0]                          # [3] Scalar+[F <- V]
    areas, dual_areas, corner_fields = triangle_geometry(topology, vertices)
    charge_map, outflow_map, extension, poisson = surface_operators(
        topology, areas, corner_fields, fractions, permittivity, conductivities)

    # Only the outer boundary voltage is held; every interior vertex voltage solves Gauss's law.
    parcels = dual_areas * fractions                            # [2] Scalar+[V]
    charge = (body_charges * parcels / parcels.sites.sum()).sum(axis=0)  # Scalar+[V]
    boundary_voltage = boundary[0]                              # Scalar+[V]
    potential = boundary_voltage + extension * poisson.solve(
        extension.adjugate() * (charge - charge_map * boundary_voltage))  # Scalar+[V]
    field = (corner_fields * (corners * potential)).sum(axis=0)  # Field+[F]
    yield potential, charge, charge / dual_areas, field.commutator(time)

    substep_dt = frame_step / n_substeps
    for target_voltage in boundary[1:]:
        previous_voltage = boundary_voltage
        for substep in range(n_substeps):
            # Equal and opposite conduction transfers conserve each isolated body's initial charge.
            charge = charge - substep_dt * (outflow_map * potential)  # Scalar+[V]
            boundary_voltage = previous_voltage + ((substep + 1) / n_substeps) * (target_voltage - previous_voltage)  # Scalar+[V]
            potential = boundary_voltage + extension * poisson.solve(
                extension.adjugate() * (charge - charge_map * boundary_voltage))  # Scalar+[V]
        field = (corner_fields * (corners * potential)).sum(axis=0)  # Field+[F]
        yield potential, charge, charge / dual_areas, field.commutator(time)


states = show_states(
    relaxation_states(topology, vertices, fractions, boundary, permittivity,
                      conductivities, body_charges, frame_step, n_substeps),
    render.relaxation_animation, topology, vertices, metal,
    name="opposite_charge_potential_notebook", duration_ms=frame_duration_ms, n_frames=n_frames);


Red and blue show positive and negative charge density, brightening towards white at the highest concentrations on a fixed scale. Gold curves trace equipotentials, and ice-blue streaks follow the electric force on a test charge, shortening and fading inside the ring cavity and conductor interiors as the internal field vanishes.

## Conserved excess charge and induced polarization

Summing vertex charges over each body's support tracks conservation alongside induced separation. Even though the positively charged rod starts with zero negative charge, the external field draws a negative lobe onto the side facing higher potential while keeping the total charge on each body constant.

In [ ]:
charges = stack([state[1] for state in states])                 # [n_frames] Scalar+[V]
net_charge = (charges[:, None] * memberships).sites.sum()       # [n_frames, 2] Scalar+
negative_charge = (where(charges < 0, -charges, 0)[:, None] * memberships).sites.sum()  # [n_frames, 2] Scalar+
save_figure(render.balance_figure(net_charge, negative_charge, frame_step), "opposite_charge_relaxation_notebook_balance");


Dashed horizontal lines show the conserved net charge on the ring in blue and on the rod in orange, while solid curves show the magnitude of negative charge gathered on each body as the applied field settles, rotates, and settles again.

## Corner charge concentration along the rod

Sharp convex corners crowd equipotentials together, concentrating surface charge above the broad polarization along flat sides.

In [ ]:
final_density = states[-1][2]                                   # Scalar+[V]
save_figure(render.corner_density_figure(vertices, final_density, scenarios.rod_centre,
                                         scenarios.rod_turn, scenarios.rod_lengths, scenarios.interface_width),
            "rod_corner_charge_density_notebook");


The orange profile traces the peak absolute charge density across the rod's boundary layer along its length, peaking sharply at the two ends marked by dotted vertical lines.

In [ ]:
# checks
assert ((net_charge - body_charges).abs() < .01).all()
assert (negative_charge[-1] > negative_charge[0]).all()
